<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day04-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 4 — In-class discussion problem (1 of 3)

Work this out **by hand in your group first** — then run the code cell
to check your answer before presenting.

## Does local alignment ever beat global on the same pair?

Align $x = \texttt{ATCG}$ against $y = \texttt{TCG}$, with
match $= +1$, mismatch $= -1$, gap $= -1$.

**As a group:**

1. Build the global (Needleman-Wunsch) DP matrix by hand.
2. Trace back from the bottom-right corner: what alignment and score do you get?
3. Now think about local (Smith-Waterman) alignment on the same $x$, $y$: where would the best-scoring region be, and what would its score be?
4. Which score is higher — global's or local's — and why?

Run the cell below to check your answers.

In [1]:
import numpy as np

def global_align(seqA, seqB, match=1.0, mismatch=-1.0, gap=-1.0):
    def score(a, b):
        return match if a == b else mismatch
    m, n = len(seqA) + 1, len(seqB) + 1
    S = np.zeros((m, n))
    trace = np.full((m, n), "", dtype=object)
    for i in range(1, m):
        S[i, 0] = i * gap
        trace[i, 0] = "up"
    for j in range(1, n):
        S[0, j] = j * gap
        trace[0, j] = "left"
    for i in range(1, m):
        for j in range(1, n):
            diag = S[i-1, j-1] + score(seqA[i-1], seqB[j-1])
            up = S[i-1, j] + gap
            left = S[i, j-1] + gap
            best = max(diag, up, left)
            S[i, j] = best
            trace[i, j] = "diag" if best == diag else ("up" if best == up else "left")
    return S, trace

def local_align(seqA, seqB, match=1.0, mismatch=-1.0, gap=-1.0):
    def score(a, b):
        return match if a == b else mismatch
    m, n = len(seqA) + 1, len(seqB) + 1
    S = np.zeros((m, n))
    for i in range(1, m):
        for j in range(1, n):
            diag = S[i-1, j-1] + score(seqA[i-1], seqB[j-1])
            up = S[i-1, j] + gap
            left = S[i, j-1] + gap
            S[i, j] = max(diag, up, left, 0.0)
    return S

def traceback_global(seqA, seqB, S, trace):
    i, j = len(seqA), len(seqB)
    outA, outB = [], []
    while i > 0 or j > 0:
        d = trace[i, j]
        if d == "diag":
            outA.append(seqA[i-1]); outB.append(seqB[j-1]); i -= 1; j -= 1
        elif d == "up":
            outA.append(seqA[i-1]); outB.append("-"); i -= 1
        else:
            outA.append("-"); outB.append(seqB[j-1]); j -= 1
    return "".join(reversed(outA)), "".join(reversed(outB))

x, y = "ATCG", "TCG"
S, trace = global_align(x, y)
print("global DP matrix:\n", S)
a, b = traceback_global(x, y, S, trace)
print("\nglobal alignment:")
print(a)
print(b)
print("global score:", S[-1, -1])

Sl = local_align(x, y)
print("\nlocal DP matrix:\n", Sl)
print("local best score:", Sl.max(), "(the pure TCG/TCG match, ignoring the leading A)")

global DP matrix:
 [[ 0. -1. -2. -3.]
 [-1. -1. -2. -3.]
 [-2.  0. -1. -2.]
 [-3. -1.  1.  0.]
 [-4. -2.  0.  2.]]

global alignment:
ATCG
-TCG
global score: 2.0

local DP matrix:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 2. 1.]
 [0. 0. 1. 3.]]
local best score: 3.0 (the pure TCG/TCG match, ignoring the leading A)


**Discussion point:** global alignment is *forced* to account for every
residue in both sequences, so the leading `A` in `x` has to go somewhere
— it gets aligned to a gap, costing -1, for a total global score of 2.
Local alignment isn't forced to explain the whole sequence: it can just
ignore the unmatched `A` entirely and report the pure `TCG`/`TCG` match,
scoring 3. Local alignment never scores *lower* than the best possible
sub-alignment global would find, because it's free to discard anything
that doesn't help.